# 03 — Inference
Load best SPOTER model and run predictions on video files.

In [ ]:
# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

import os, sys
PROJECT_ROOT = '/content/drive/MyDrive/isl_project'
sys.path.insert(0, f'{PROJECT_ROOT}/src')

# Install ONLY packages not preinstalled on Colab
%pip install -q mediapipe==0.10.21 opencv-python-headless==4.11.0.86 scipy==1.13.0 pyyaml>=6.0 tqdm>=4.65.0 scikit-learn>=1.3.0 seaborn>=0.12.0

In [ ]:
# Load model
import torch, json, yaml, numpy as np
from models.spoter import SPOTER
from utils import predict_video

SPLITS_DIR = f'{PROJECT_ROOT}/data/splits'
CKPT = f'{PROJECT_ROOT}/checkpoints/spoter/best.pt'
CONFIG = f'{PROJECT_ROOT}/configs/spoter.yaml'
MODEL_PATH = f'{PROJECT_ROOT}/models/holistic_landmarker.task'

with open(CONFIG) as f:
    cfg = yaml.safe_load(f)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = SPOTER(
    input_dim=cfg['input_dim'], d_model=cfg['d_model'],
    nhead=cfg['nhead'], num_encoder_layers=cfg['num_encoder_layers'],
    dim_feedforward=cfg['dim_feedforward'],
    num_classes=cfg['num_classes'], dropout=0.0
)
ckpt = torch.load(CKPT, map_location=device)
model.load_state_dict(ckpt['model_state_dict'])
model.to(device).eval()

mean = np.load(f'{SPLITS_DIR}/mean.npy')
std = np.load(f'{SPLITS_DIR}/std.npy')

with open(f'{SPLITS_DIR}/label_map.json') as f:
    label_map = json.load(f)
label_names = [k for k, _ in sorted(label_map.items(), key=lambda x: x[1])]

print(f'Model loaded. {len(label_names)} classes.')

In [ ]:
# Predict on a video file
VIDEO_PATH = '/content/test_sign.mp4'  # upload your video
results = predict_video(model, VIDEO_PATH, MODEL_PATH, mean, std, label_names, device, top_k=5)
print('Top-5 Predictions:')
for rank, (label, conf) in enumerate(results, 1):
    print(f'  {rank}. {label}: {conf*100:.1f}%')

In [ ]:
# Upload and test from Colab file picker
from google.colab import files
uploaded = files.upload()
for fname in uploaded:
    results = predict_video(model, fname, MODEL_PATH, mean, std, label_names, device)
    print(f'\nPredictions for {fname}:')
    for r, (l, c) in enumerate(results, 1):
        print(f'  {r}. {l}: {c*100:.1f}%')